In [ ]:
!pip install groq -q

import json
import time
from groq import Groq
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 4.8 MB/s eta 0:00:00


In [ ]:
GROQ_API_KEY = "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"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

Claim-Only Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    # premise = item.get("premise", "")
    # argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "claim": claim,
                "model": MODEL_ID,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(1)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: Making museums free of charge aligns with the principle of fair equality of opportunity, as it allows all citizens, regardless of their economic background, to access cultural and educational resources. From behind the veil of ignorance, individuals would likely agree to such a policy, as it promotes the common good and benefits the least advantaged by enriching their social and cultural lives. By removing financial barriers, we ensure that the distribution of cultural goods is not unduly influenced by morally arbitrary factors like wealth. This policy contributes to a more just and equitable society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against
Argument: Making all museums free 

In [ ]:
import os
import json

output_folder = '/content/drive/MyDrive/masterthesis/llama-4-maverick-17b-128e-instruct/one_shot'

os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, 'Claim_only_output.json')

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

Claim+Premise Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    # argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"
"Premise: {premise}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(1)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: Making museums free of charge aligns with the principle of fair equality of opportunity, as it allows all citizens, regardless of their economic background, to access and benefit from cultural heritage. From behind the veil of ignorance, individuals would likely agree to a system where basic cultural institutions are accessible to all, promoting social cohesion and mutual understanding. This policy also benefits the least advantaged by providing them with opportunities for education and enrichment that they might otherwise be unable to afford. By doing so, it contributes to a more just and equitable society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 1 | Persona: Libertarian economist
Response: Stance: Against
Argument: Making museums fr

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-maverick-17b-128e-instruct/one_shot/Claim_Premise_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)

Claim + Premise + Argumentation Prompt

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key="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")
MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"

with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    debate_data = json.load(f)

PERSONAS = [
    "Rawlsian philosopher",
    "Libertarian economist",
    "Utilitarian ethicist",
    "Conservative political theorist"
]

results = []
few_shot_examples = {
    "Rawlsian philosopher": """
Example:
Claim: "Governments should increase redistribution to reduce economic inequality."
Persona Output:
Stance: For
Argument: Justice requires structuring society so that inequalities work to benefit the least advantaged. From behind the veil of ignorance, no one would accept institutions that allow severe distributive gaps. Redistribution ensures fair equality of opportunity and protects citizens from morally arbitrary disadvantages.
""",

    "Libertarian economist": """
Example:
Claim: "The government should impose strict regulations on the tech industry."
Persona Output:
Stance: Against
Argument: Markets function best when voluntary exchange—not state intervention—drives innovation. Heavy regulation restricts entrepreneurial freedom, distorts incentives, and stifles competition. Individuals and firms, not bureaucrats, are the most efficient coordinators of economic activity.
""",

    "Utilitarian ethicist": """
Example:
Claim: "Public funds should prioritize pandemic preparedness over military spending."
Persona Output:
Stance: For
Argument: Allocating resources to maximize overall well-being requires reducing preventable suffering and death. Pandemic preparedness generates substantial net benefit by protecting large populations. Utility calculations clearly favor investments that minimize harm and produce the greatest total welfare.
""",

    "Conservative political theorist": """
Example:
Claim: "Long-standing cultural institutions should be rapidly restructured to meet modern values."
Persona Output:
Stance: Against
Argument: Social stability depends on institutions that evolve gradually through accumulated wisdom. Abrupt transformation risks undermining cohesion and disregarding proven traditions. Conservatism urges caution: reform must build on continuity, not rupture it.
"""
}

task_counter = 0
total_tasks = len(debate_data) * len(PERSONAS)

for item in debate_data:
    item_index = item.get("index")
    claim = item.get("claim", "")
    premise = item.get("premise", "")
    argumentation = item.get("argumentation", "")

    for persona in PERSONAS:
        task_counter += 1
        user_prompt = f"""
Below is a demonstration of how you should answer as a {persona}:

{few_shot_examples[persona]}

Now switch topics. Here is the new claim:
"Claim: {claim}"
"Premise: {premise}"
"Argumentation: {argumentation}"

Your task:
1. Decide whether the stance is For or Against.
2. Then give 3–4 persona-style sentences reflecting the worldview of a {persona}.

Format:
Stance: <For/Against>
Argument: <3–4 sentences>
"""
        try:
            start_time = time.time()
            chat_completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {persona}. Always respond in this persona."},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=500,
                temperature=0.7
            )
            content = chat_completion.choices[0].message.content.strip()
            duration = time.time() - start_time
            results.append({
                "index": item_index,
                "persona": persona,
                "response": content
            })

            print("-" * 60)
            print(f"Processing Item Index: {item_index} | Persona: {persona}")
            print(f"Response: {content}")
            print("-" * 60)
            print()
        except Exception as e:
            print(f"Error - Item {item_index}, Persona {persona}: {str(e)}")
            time.sleep(5)
        time.sleep(1)

------------------------------------------------------------
Processing Item Index: 1 | Persona: Rawlsian philosopher
Response: Stance: For
Argument: From behind the veil of ignorance, individuals would agree to a society where cultural institutions like museums are accessible to all, as this ensures fair equality of opportunity and promotes the common good. Making museums free of charge is a step towards creating a more just society, where the least advantaged have access to the same cultural and educational resources as the more advantaged. This aligns with the principle of distributive justice, as it redistributes the benefit of cultural heritage to all citizens, regardless of their economic status. By making museums free, we are also promoting the development of citizens' capabilities and fostering a more inclusive and equitable society.
------------------------------------------------------------

------------------------------------------------------------
Processing Item Index: 

In [ ]:
with open('/content/drive/MyDrive/masterthesis/llama-4-maverick-17b-128e-instruct/one_shot/Claim_Premise_Argumentation_output.json', 'w', encoding='utf-8') as f:
  json.dump(results, f, ensure_ascii=False, indent=4)